# Tutorial 4 — Gradient Hooks & Distribution Monitoring

**Series:** Training Language Models from Scratch: A Hacker's Guide  
**Part II — Debugging & Observability**  
**Follows:** Tutorial 3 (Tokenization)  
**Precedes:** Tutorial 5 (Real-Time Training Dashboards)

---

## What This Tutorial Covers

A training run is a black box by default. Loss goes down — or it doesn't — and you have almost no visibility into why. This tutorial gives you the instrumentation to open that box.

The central diagnostic is the **gradient-to-weight ratio**: $\rho_l = \|\nabla W_l\| / \|W_l\|$ for each layer $l$. This single number tells you whether each layer is learning at a healthy rate, stuck, or about to explode. But to use it well you need to understand what controls it — and there are exactly five levers:

1. **Initialization** — sets the starting $\|W\|$ (the denominator)
2. **Layer Norm** — keeps activation scale bounded so gradients don't blow up through saturated regions
3. **Learning rate and schedule** — directly scales $\|\Delta W\| = \text{lr} \cdot \|\nabla W\|$
4. **Gradient clipping** — hard ceiling on $\|\nabla W\|$ globally
5. **Residual stream scaling** — the $1/\sqrt{2L}$ init trick from Tutorial 2 that keeps per-layer contributions bounded as depth grows

This tutorial builds a `GradientMonitor` that records $\rho_l$ in real time, then breaks each lever one at a time on a live training run and shows you exactly what breaks on the chart. By the end you will be able to look at a gradient-to-weight ratio plot and diagnose which lever is misconfigured.

The hooks covered:

- `register_forward_hook` — captures activations as they flow forward
- `register_backward_hook` — captures gradients as they flow backward  
- `register_full_backward_hook` — full input/output gradient tensors per module
- `register_hook` on parameters — captures parameter gradients directly

---

## 1. The Hook API

PyTorch hooks are callbacks that fire at specific points in the forward or backward pass. They do not modify the computation — they observe it.

### Forward hooks

`module.register_forward_hook(fn)` *fires* immediately after `module.forward()` returns, with signature:

```python
fn(module, input, output) -> None
```

`input` is a tuple of the module's inputs. `output` is the module's output tensor (or tuple). This is where you capture activations:

In [ ]:
import torch
import torch.nn as nn

layer = nn.Linear(64, 64)
activations = {}

def forward_hook(module, input, output):
    # output: (batch, 64) — the post-linear activations
    activations['linear'] = output.detach()   # detach — we don't want this in the graph

handle = layer.register_forward_hook(forward_hook)

x = torch.randn(8, 64)
_ = layer(x)
print(activations['linear'].shape)   # torch.Size([8, 64])

handle.remove()   # always remove hooks when done — they hold references

[**Always call `.detach()` inside hooks.**]{.mark} Without it, you keep a reference to the computation graph node, preventing it from being freed after `.backward()`. This is a **memory leak** that grows every step.

**Always store the handle and call `.remove()`.** Hooks are stored in the module and fire on every forward pass. If you register the same hook multiple times (e.g., inside a training loop), you accumulate redundant hooks that all fire and slow down training.

### Backward hooks

`module.register_full_backward_hook(fn)` fires during the backward pass with signature:

```python
fn(module, grad_input, grad_output) -> None
```

`grad_output` is the gradient flowing *into* this module from downstream (upstream in the network, downstream in the graph). `grad_input` is the gradient flowing *out* of this module to its inputs. For diagnosing gradient flow, `grad_output` is what you want — it tells you what gradient signal the module received:

In [ ]:
gradients = {}

def backward_hook(module, grad_input, grad_output):
    # grad_output: tuple of gradients w.r.t. module's outputs
    # We want grad_output[0] — gradient of loss w.r.t. this module's output
    if grad_output[0] is not None:
        gradients['linear'] = grad_output[0].detach()

handle = layer.register_full_backward_hook(backward_hook)

x = torch.randn(8, 64, requires_grad=True)
out = layer(x)
out.sum().backward()

print(gradients['linear'].shape)   # torch.Size([8, 64])
handle.remove()

### Parameter gradient hooks

For per-parameter gradient inspection, register a hook directly on the parameter tensor:

In [ ]:
layer = nn.Linear(64, 64)
param_grads = {}

def param_grad_hook(grad):
    param_grads['weight'] = grad.detach().clone()
    return grad   # must return grad (or None to zero it)

handle = layer.weight.register_hook(param_grad_hook)

x = torch.randn(8, 64)
layer(x).sum().backward()
print(param_grads['weight'].norm().item())
handle.remove()

This fires after the gradient is accumulated into `.grad` but before the optimizer reads it — giving you a clean view of the raw gradient before any optimizer transformations.

---

## 2. The `GradientMonitor` Class

Now we build the real instrument. The `GradientMonitor` attaches to every named module in a model, records statistics every step, and computes the gradient-to-weight ratio:

In [ ]:
import torch
import torch.nn as nn
from collections import defaultdict
from dataclasses import dataclass, field
import math

@dataclass
class LayerStats:
    """Statistics for one layer at one training step."""
    step:           int
    layer_name:     str
    # Activation statistics (forward pass)
    act_mean:       float = 0.0
    act_std:        float = 0.0
    act_frac_zero:  float = 0.0   # fraction of zero activations (dead neurons)
    # Gradient statistics (backward pass)
    grad_mean:      float = 0.0
    grad_std:       float = 0.0
    grad_l2:        float = 0.0   # gradient L2 norm
    grad_max:       float = 0.0   # max absolute gradient
    # Weight statistics
    weight_l2:      float = 0.0   # weight L2 norm
    # The key ratio
    grad_weight_ratio: float = 0.0  # grad_l2 / weight_l2


class GradientMonitor:
    """
    Attaches forward and backward hooks to every named module in a model.
    Records per-layer activation and gradient statistics every training step.

    Usage:
        monitor = GradientMonitor(model)
        monitor.attach()

        for step, batch in enumerate(dataloader):
            loss = train_step(batch)
            monitor.record(step)         # call after loss.backward()

        monitor.detach()
        df = monitor.to_dataframe()      # pandas DataFrame of all stats
    """

    def __init__(self, model: nn.Module, layers_to_watch: list[str] = None):
        """
        Args:
            model: the model to instrument
            layers_to_watch: list of module name substrings to watch.
                             If None, watches all nn.Linear layers.
        """
        self.model          = model
        self.layers_to_watch = layers_to_watch
        self.history: list[LayerStats] = []
        self._handles = []

        # Temporary storage filled during forward/backward
        self._act_buffer:  dict[str, torch.Tensor] = {}
        self._grad_buffer: dict[str, torch.Tensor] = {}

    def _should_watch(self, name: str, module: nn.Module) -> bool:
        if self.layers_to_watch is not None:
            return any(pat in name for pat in self.layers_to_watch)
        return isinstance(module, nn.Linear)

    def attach(self):
        """Register hooks on all watched modules."""
        for name, module in self.model.named_modules():
            if not self._should_watch(name, module):
                continue

            # Capture name in closure
            def make_forward_hook(n):
                def hook(mod, inp, out):
                    self._act_buffer[n] = out.detach().float()
                return hook

            def make_backward_hook(n):
                def hook(mod, grad_in, grad_out):
                    if grad_out[0] is not None:
                        self._grad_buffer[n] = grad_out[0].detach().float()
                return hook

            self._handles.append(
                module.register_forward_hook(make_forward_hook(name))
            )
            self._handles.append(
                module.register_full_backward_hook(make_backward_hook(name))
            )

        print(f"GradientMonitor: attached to {len(self._handles)//2} layers")

    def detach(self):
        """Remove all hooks."""
        for h in self._handles:
            h.remove()
        self._handles.clear()
        print("GradientMonitor: all hooks removed")

    def record(self, step: int):
        """
        Call after loss.backward(). Reads buffers, computes stats,
        appends to history. Clears buffers for next step.
        """
        for name in self._act_buffer:
            act  = self._act_buffer.get(name)
            grad = self._grad_buffer.get(name)

            # --- Activation stats ---
            act_mean     = act.mean().item()     if act  is not None else 0.0
            act_std      = act.std().item()      if act  is not None else 0.0
            act_frac_zero = (act == 0).float().mean().item() if act is not None else 0.0

            # --- Gradient stats ---
            grad_mean = grad.mean().item()  if grad is not None else 0.0
            grad_std  = grad.std().item()   if grad is not None else 0.0
            grad_l2   = grad.norm().item()  if grad is not None else 0.0
            grad_max  = grad.abs().max().item() if grad is not None else 0.0

            # --- Weight stats (from the module itself) ---
            weight_l2 = 0.0
            for n, mod in self.model.named_modules():
                if n == name and hasattr(mod, 'weight') and mod.weight is not None:
                    weight_l2 = mod.weight.norm().item()
                    break

            # --- The ratio ---
            ratio = grad_l2 / (weight_l2 + 1e-8)

            self.history.append(LayerStats(
                step=step,
                layer_name=name,
                act_mean=act_mean,
                act_std=act_std,
                act_frac_zero=act_frac_zero,
                grad_mean=grad_mean,
                grad_std=grad_std,
                grad_l2=grad_l2,
                grad_max=grad_max,
                weight_l2=weight_l2,
                grad_weight_ratio=ratio,
            ))

        self._act_buffer.clear()
        self._grad_buffer.clear()

    def to_dataframe(self):
        """Convert history to a pandas DataFrame for analysis."""
        import pandas as pd
        return pd.DataFrame([vars(s) for s in self.history])

    def latest_ratios(self) -> dict[str, float]:
        """Return the most recent grad/weight ratio for each layer."""
        latest = {}
        for stat in reversed(self.history):
            if stat.layer_name not in latest:
                latest[stat.layer_name] = stat.grad_weight_ratio
            if len(latest) == len(set(s.layer_name for s in self.history)):
                break
        return latest

    def warn_if_unhealthy(self, lo: float = 1e-4, hi: float = 1e-1):
        """
        Print warnings for layers whose grad/weight ratio is outside [lo, hi].
        Healthy range for most architectures: ~1e-3 to ~1e-2.
        """
        ratios = self.latest_ratios()
        for name, ratio in ratios.items():
            if ratio < lo:
                print(f"  ⚠ VANISHING  {name:50s}  ratio={ratio:.2e}  (< {lo:.0e})")
            elif ratio > hi:
                print(f"  ⚠ EXPLODING  {name:50s}  ratio={ratio:.2e}  (> {hi:.0e})")

---

## 3. The Gradient-to-Weight Ratio: What Healthy Looks Like

Before breaking things, establish the baseline. Train the nano GPT from Tutorial 2 for 200 steps with correct settings and record the ratio:

In [ ]:
import torch
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import numpy as np

# --- Setup ---
from tutorial_02 import GPT, NanoGPTConfig   # your implementations
from tutorial_03 import Tokenizer

config = NanoGPTConfig()
model  = GPT(config)
tok    = Tokenizer.load('nano_tokenizer.json')

# Tiny dataset — TinyShakespeare, packed into fixed-length chunks
text   = open('tinyshakespeare.txt').read()
data   = torch.tensor(tok.encode(text), dtype=torch.long)
block  = config.max_seq_len

def get_batch(batch_size=8):
    ix = torch.randint(len(data) - block, (batch_size,))
    x  = torch.stack([data[i   : i+block  ] for i in ix])
    y  = torch.stack([data[i+1 : i+block+1] for i in ix])
    return x, y

optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=0.1)
monitor   = GradientMonitor(model)
monitor.attach()

# --- Training loop ---
losses = []
for step in range(200):
    x, y = get_batch()
    _, loss = model(x, y)

    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()

    monitor.record(step)
    losses.append(loss.item())

    if step % 50 == 0:
        print(f"step {step:3d}  loss={loss.item():.4f}")
        monitor.warn_if_unhealthy()

monitor.detach()
df = monitor.to_dataframe()

Now plot the ratio across layers and steps:

In [ ]:
def plot_ratio_heatmap(df, title="Gradient-to-Weight Ratio"):
    """
    Heatmap: x=step, y=layer, color=log10(ratio).
    Healthy range (green): 1e-3 to 1e-2.
    Too small (blue): vanishing. Too large (red): exploding.
    """
    layers = df['layer_name'].unique()
    steps  = df['step'].unique()

    # Pivot to (layers × steps) matrix
    matrix = np.full((len(layers), len(steps)), np.nan)
    layer_idx = {l: i for i, l in enumerate(layers)}
    step_idx  = {s: i for i, s in enumerate(steps)}

    for _, row in df.iterrows():
        i = layer_idx[row['layer_name']]
        j = step_idx[row['step']]
        ratio = row['grad_weight_ratio']
        matrix[i, j] = math.log10(ratio + 1e-10)

    fig, ax = plt.subplots(figsize=(14, 6))
    im = ax.imshow(
        matrix,
        aspect='auto',
        cmap='RdYlGn',        # red=bad, yellow=ok, green=good
        vmin=-5,              # log10(1e-5) — very vanishing
        vmax=-1,              # log10(1e-1) — very large
        origin='upper'
    )
    ax.set_yticks(range(len(layers)))
    ax.set_yticklabels(layers, fontsize=8)
    ax.set_xlabel('Training Step')
    ax.set_ylabel('Layer')
    ax.set_title(title)
    plt.colorbar(im, ax=ax, label='log₁₀(‖∇W‖ / ‖W‖)')
    plt.tight_layout()
    plt.savefig(f'{title.lower().replace(" ", "_")}.png', dpi=150)
    plt.show()

plot_ratio_heatmap(df, title="Baseline — Healthy Training")

[[A healthy chart is uniformly green across all layers and all steps. The ratio should be roughly in $[10^{-3}, 10^{-2}]$]{.mark} — meaning each update moves the weights by about 0.1% to 1% of their current magnitude per step. This is the reference you will compare every broken run against.

---

## 4. The Five Levers

Now we break each lever one at a time and watch what happens to the chart. Each experiment runs the same 200 steps on the same data, with one setting changed.

### Lever 1: Initialization

**What it controls:** The starting value of $\|W\|$ (the denominator of $\rho$). If weights are initialized too small, $\|W\|$ is small and $\rho$ appears healthy even if gradients are tiny in absolute terms — but the weights will stay small because each update is also tiny. If weights are initialized too large, activations saturate immediately and gradients through the saturating nonlinearity vanish.

In [ ]:
def run_with_init(std: float, label: str, steps: int = 200):
    model = GPT(config)

    # Override initialization: all linear weights ~ N(0, std²)
    for module in model.modules():
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=std)
            if module.bias is not None:
                nn.init.zeros_(module.bias)

    optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)
    monitor   = GradientMonitor(model)
    monitor.attach()

    for step in range(steps):
        x, y = get_batch()
        _, loss = model(x, y)
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        monitor.record(step)

    monitor.detach()
    return monitor.to_dataframe().assign(experiment=label)

df_init_ok   = run_with_init(std=0.02,  label='init std=0.02 (correct)')
df_init_tiny = run_with_init(std=0.001, label='init std=0.001 (too small)')
df_init_huge = run_with_init(std=0.5,   label='init std=0.5 (too large)')

**What you see:**
- `std=0.02`: Green heatmap, ratio stable around $10^{-2.5}$
- `std=0.001`: Ratio appears artificially high ($\rho = \|\nabla W\| / \|W\|$ — tiny $\|W\|$ inflates $\rho$), but absolute gradient norms are normal; weights barely move in absolute terms
- `std=0.5`: Ratio is tiny across all layers — large $\|W\|$ deflates $\rho$; the softmax inside attention saturates immediately (large logits → peaky distribution → near-zero gradient), and loss barely moves

[[The lesson: initialization sets the scale of the system. Both the numerator and denominator of $\rho$ depend on it, so the ratio alone is not enough — always plot absolute gradient norms alongside the ratio.]{.underline}

### Lever 2: Layer Norm vs No Layer Norm

**What it controls:** Whether activation magnitudes are bounded across layers. Without LN, activations grow or shrink as they propagate through layers — gradients follow. With LN, each layer sees normalized inputs regardless of what earlier layers did.

In [ ]:
class TransformerBlockNoLN(nn.Module):
    """TransformerBlock with LayerNorm removed."""
    def __init__(self, d_model, n_heads, n_layers):
        super().__init__()
        self.attn = MultiHeadAttention(d_model, n_heads)
        self.ffn  = FFN(d_model)

    def forward(self, x, mask=None):
        # No LayerNorm — raw residual connections only
        attn_out, _ = self.attn(x, mask=mask)
        x = x + attn_out
        x = x + self.ffn(x)
        return x

class GPTNoLN(GPT):
    """GPT with LayerNorm removed from all blocks."""
    def __init__(self, config):
        super().__init__(config)
        # Replace blocks with no-LN versions
        self.blocks = nn.ModuleList([
            TransformerBlockNoLN(config.d_model, config.n_heads, config.n_layers)
            for _ in range(config.n_layers)
        ])
        # Remove final layer norm
        self.norm_out = nn.Identity()

def run_with_model(model_cls, label, steps=200):
    model     = model_cls(config)
    optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)
    monitor   = GradientMonitor(model)
    monitor.attach()

    for step in range(steps):
        x, y = get_batch()
        _, loss = model(x, y)
        if torch.isnan(loss):
            print(f"  NaN at step {step} — stopping")
            break
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        monitor.record(step)

    monitor.detach()
    return monitor.to_dataframe().assign(experiment=label)

df_with_ln = run_with_model(GPT,     label='With LayerNorm')
df_no_ln   = run_with_model(GPTNoLN, label='No LayerNorm')

**What you see:**
- With LN: Ratio is uniform across layers — LN normalizes each layer's input to roughly unit variance, so each layer sees the same activation scale
- Without LN: Ratio degrades with depth — early layers show vanishing ratios, final layers show large ratios; the activation scale compounds across layers and gradients flow unevenly; the run often produces NaN within 100 steps

[[This is the precise quantitative reason LN matters for deep models, and why BN cannot substitute: BN normalizes across the batch (wrong dimension), while LN normalizes across features (correct dimension for language models).]{.mark}

### Lever 3: Learning Rate and Schedule

**What it controls:** $\|\Delta W\| = \text{lr} \cdot \|\nabla W\|$. The ratio $\rho = \|\nabla W\| / \|W\|$ does not include the LR — but the *effective* update ratio $\|\Delta W\| / \|W\| = \text{lr} \cdot \rho$ does. A too-large LR makes the effective ratio large; updates overshoot and destabilize. A too-small LR makes the effective ratio tiny; the model barely moves.

In [ ]:
def run_with_lr(lr: float, label: str, steps: int = 200):
    model     = GPT(config)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.1)
    monitor   = GradientMonitor(model)
    monitor.attach()

    losses = []
    for step in range(steps):
        x, y = get_batch()
        _, loss = model(x, y)
        if torch.isnan(loss):
            print(f"  NaN loss at step {step}")
            break
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        monitor.record(step)
        losses.append(loss.item())

    monitor.detach()
    df = monitor.to_dataframe().assign(experiment=label)
    df['loss'] = [losses[min(r.step, len(losses)-1)]
                  for r in monitor.history[:len(df)]]
    return df

df_lr_good = run_with_lr(3e-4,  label='lr=3e-4 (good)')
df_lr_low  = run_with_lr(1e-6,  label='lr=1e-6 (too low)')
df_lr_high = run_with_lr(3e-2,  label='lr=3e-2 (too high)')

**What you see on the ratio chart:**
- The raw gradient-to-weight ratio $\rho$ looks similar for all three LRs — it is independent of LR. The ratio chart alone cannot distinguish a good LR from a bad one.
- The loss chart tells the story: `lr=1e-6` barely moves; `lr=3e-2` spikes and may diverge

This is an important lesson: [**the gradient-to-weight ratio is not sufficient on its own**]{.mark}. You must plot it alongside the loss curve and the effective update ratio $\text{lr} \cdot \rho$. The dashboard in Tutorial 5 shows all three.

The warmup schedule effect is visible differently: during warmup, LR rises from 0, so the effective update ratio rises too. After warmup the ratio stabilizes. A missing warmup (jumping straight to full LR) often produces a spike in gradient norms in the first few steps — the model makes large updates before the optimizer's momentum has settled.

### Lever 4: Gradient Clipping

**What it controls:** A hard ceiling on the global gradient norm $\|g\| = \sqrt{\sum_l \|\nabla W_l\|^2}$. When $\|g\| > c$, all gradients are rescaled by $c / \|g\|$. This prevents individual large-gradient events (spiky batches, bad data) from destabilizing training.

In [ ]:
def run_with_clip(clip: float, label: str, steps: int = 200):
    model     = GPT(config)
    optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)
    monitor   = GradientMonitor(model)
    monitor.attach()

    grad_norms = []
    for step in range(steps):
        x, y = get_batch()
        _, loss = model(x, y)
        optimizer.zero_grad()
        loss.backward()

        # Compute norm BEFORE clipping
        raw_norm = torch.nn.utils.clip_grad_norm_(
            model.parameters(), clip
        ).item()   # returns the norm before clipping
        grad_norms.append(raw_norm)

        optimizer.step()
        monitor.record(step)

    monitor.detach()
    return monitor.to_dataframe().assign(experiment=label), grad_norms

df_clip1,  norms_clip1  = run_with_clip(1.0,  'clip=1.0 (standard)')
df_clip01, norms_clip01 = run_with_clip(0.1,  'clip=0.1 (aggressive)')
df_noclip, norms_noclip = run_with_clip(1e9,  'no clipping')

**What you see:**
- The ratio chart for `clip=1.0` looks stable — clipping fires occasionally (when `raw_norm > 1.0`) and prevents outlier steps from corrupting the weight scale
- `clip=0.1` produces a ratio chart that looks fine but training is slow — clipping fires on most steps, reducing effective gradient signal
- `no clipping` may show sudden ratio spikes — a batch of unusually large loss (e.g., a very long or repetitive sequence) produces a gradient spike that destabilizes several subsequent steps

[The pre-clip gradient norm is the key signal.]{.underline} If it is consistently near `clip`, your model is perpetually constrained — either reduce LR or increase clip. If it is never near `clip`, the clip threshold is irrelevant and can be raised. If it spikes occasionally to 10× the clip value, clipping is doing its job.

In [ ]:
# Visualize pre-clip gradient norm distribution
plt.figure(figsize=(10, 4))
plt.plot(norms_clip1,  label='raw norm (clip=1.0)')
plt.plot(norms_noclip, label='raw norm (no clip)', alpha=0.6)
plt.axhline(1.0, color='red', linestyle='--', label='clip threshold')
plt.xlabel('Step')
plt.ylabel('Global Gradient Norm')
plt.title('Pre-Clip Gradient Norm — clip=1.0 vs no clipping')
plt.legend()
plt.yscale('log')
plt.savefig('gradient_norm_clipping.png', dpi=150)
plt.show()

### Lever 5: Residual Stream Scaling

**What it controls:** How much each residual block contributes to the stream at initialization. Without the $1/\sqrt{2L}$ scaling on output projections, the stream variance grows as $O(L)$ — for 12 layers it is 12× larger than intended, causing gradient norms to vary systematically with depth (deeper layers contribute more to the stream and receive larger gradients).

This is the lever from Tutorial 2 Section 9 — now we see its effect quantitatively:

In [ ]:
def make_model_no_residual_scaling():
    model = GPT(config)
    # Remove the residual scaling — keep standard init
    for name, module in model.named_modules():
        if isinstance(module, nn.Linear) and ('W_o' in name or 'fc2' in name):
            nn.init.normal_(module.weight, 0.0, 0.02)  # reset to standard init
    return model

def make_model_with_residual_scaling():
    model = GPT(config)
    # Already applied in GPT.__init__ via apply_residual_scaling()
    return model

def run_model(model, label, steps=200):
    optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)
    monitor   = GradientMonitor(model)
    monitor.attach()
    for step in range(steps):
        x, y = get_batch()
        _, loss = model(x, y)
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        monitor.record(step)
    monitor.detach()
    return monitor.to_dataframe().assign(experiment=label)

df_scaled    = run_model(make_model_with_residual_scaling(), 'With 1/√(2L) scaling')
df_unscaled  = run_model(make_model_no_residual_scaling(),   'Without scaling')

**What you see:**
- With scaling: Per-layer ratios are roughly uniform across depth — each layer contributes a similar magnitude update to the stream
- Without scaling: Ratio degrades monotonically with depth — the final layers have much larger ratios than early layers because they are closer to the loss and the stream variance compounds; early layers barely receive gradient signal in the first steps, delaying when they start learning

For a 6-layer nano model the effect is moderate. At 96 layers (GPT-3 scale) without this scaling, [the first 20 layers effectively do not train for hundreds of steps — a catastrophic waste of compute]{.underline}.

---

## 5. The Combined Chart

Put all five levers on one figure — each as a separate subplot showing the mean ratio across all layers over time:

In [ ]:
import pandas as pd

def mean_ratio_per_step(df):
    return df.groupby('step')['grad_weight_ratio'].mean()

experiments = {
    'Baseline (all correct)':    df,
    'Init std=0.001 (too small)': df_init_tiny,
    'Init std=0.5 (too large)':   df_init_huge,
    'No LayerNorm':               df_no_ln,
    'No residual scaling':        df_unscaled,
}

fig, axes = plt.subplots(len(experiments), 1, figsize=(12, 14), sharex=True)

for ax, (label, df_exp) in zip(axes, experiments.items()):
    ratio = mean_ratio_per_step(df_exp)
    ax.semilogy(ratio.index, ratio.values, linewidth=2)
    ax.axhline(1e-3, color='green',  linestyle='--', alpha=0.5, label='healthy lo')
    ax.axhline(1e-2, color='green',  linestyle='--', alpha=0.5, label='healthy hi')
    ax.set_ylabel('mean ρ')
    ax.set_title(label)
    ax.set_ylim(1e-6, 1e0)
    ax.legend(loc='upper right', fontsize=8)

axes[-1].set_xlabel('Training Step')
fig.suptitle('Five Levers on the Gradient-to-Weight Ratio', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('five_levers.png', dpi=150, bbox_inches='tight')
plt.show()

This is the chart you will return to every time you start a new training run. Five subplots, one per lever, each showing what the ratio chart looks like when that lever is misconfigured. The baseline is the reference — everything else is a deviation pattern you now recognize.

---

## 6. Dead Neurons via Activation Hooks

The activation hook gives us more than gradient flow — it tells us about dead neurons.

A **dead neuron** is a ReLU (or GeLU) unit that outputs zero for every input in a batch — its weights have been pushed into the permanently-negative region and it no longer participates in computation. The gradient through a dead ReLU is exactly zero (the derivative of `max(0, x)` at `x < 0` is 0), so the neuron cannot recover via gradient descent.

We detect this via the `act_frac_zero` field in `LayerStats` — the fraction of activation values that are exactly zero in the forward pass:

In [ ]:
def plot_dead_neurons(df, threshold=0.5):
    """
    Plot fraction of zero activations per layer over training.
    A layer with frac_zero > threshold has a dead neuron problem.
    """
    fig, ax = plt.subplots(figsize=(12, 5))

    for layer_name, group in df.groupby('layer_name'):
        if 'ffn' not in layer_name:   # focus on FFN layers where ReLU/GeLU lives
            continue
        ax.plot(
            group['step'],
            group['act_frac_zero'],
            label=layer_name,
            alpha=0.7
        )

    ax.axhline(threshold, color='red', linestyle='--', label=f'dead threshold ({threshold})')
    ax.set_xlabel('Step')
    ax.set_ylabel('Fraction Zero Activations')
    ax.set_title('Dead Neuron Monitor — FFN Layers')
    ax.legend(fontsize=8, ncol=2)
    plt.tight_layout()
    plt.savefig('dead_neurons.png', dpi=150)
    plt.show()

plot_dead_neurons(df)

**What triggers dead neurons:**
- Learning rate too high: weights get pushed far negative in the first few steps
- Gradient clipping too aggressive: the model cannot recover neurons once they die
- Bad initialization: some neurons initialized in the dead zone and never escape

**GELU vs ReLU:** GELU does not produce exactly-zero activations (it has a smooth negative region)[^gelu_zero], so `act_frac_zero` will always be 0 for GELU.

[^gelu_zero]: GELU(x) = x·Φ(x) where Φ is the CDF of the standard normal. Since Φ(x) > 0 for all finite x, the output is exactly zero only when x = 0 — which essentially never occurs with floating-point arithmetic. For GELU networks, use the fraction of *near-zero* activations (e.g., `abs(act) < 0.01`) as the dead-neuron proxy:

```python
# In the GradientMonitor, for GELU networks:
act_frac_zero = (act.abs() < 0.01).float().mean().item()
```

---

## 7. A Practical Monitoring Workflow

Here is how to use the monitor in a real training run — lightweight enough to leave on throughout training:

In [ ]:
class LightweightMonitor:
    """
    A lower-overhead version of GradientMonitor for production training runs.
    Records only the global gradient norm and per-layer ratio at a fixed interval.
    """
    def __init__(self, model: nn.Module, log_every: int = 50):
        self.model     = model
        self.log_every = log_every
        self.history   = []

    def log(self, step: int, loss: float):
        if step % self.log_every != 0:
            return

        record = {'step': step, 'loss': loss}

        # Global gradient norm (cheap — already computed for clipping)
        total_norm = 0.0
        for p in self.model.parameters():
            if p.grad is not None:
                total_norm += p.grad.norm().item() ** 2
        record['global_grad_norm'] = total_norm ** 0.5

        # Per-layer ratio (only for named Linear layers)
        ratios = {}
        for name, module in self.model.named_modules():
            if isinstance(module, nn.Linear) and module.weight.grad is not None:
                g = module.weight.grad.norm().item()
                w = module.weight.norm().item()
                ratios[name] = g / (w + 1e-8)
        record['ratios'] = ratios
        record['mean_ratio'] = sum(ratios.values()) / max(len(ratios), 1)

        self.history.append(record)

        # Print summary
        lo = min(ratios.values()) if ratios else 0
        hi = max(ratios.values()) if ratios else 0
        print(
            f"step {step:5d}  loss={loss:.4f}  "
            f"grad_norm={record['global_grad_norm']:.3f}  "
            f"ratio=[{lo:.2e}, {hi:.2e}]"
        )

# Usage inside the training loop:
monitor = LightweightMonitor(model, log_every=50)

for step in range(max_steps):
    x, y   = get_batch()
    _, loss = model(x, y)
    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()

    monitor.log(step, loss.item())   # one call, no hooks needed

The `LightweightMonitor` reads `.grad` directly after `.backward()` — no hooks, no closures, no risk of memory leaks. The full `GradientMonitor` with hooks is for debugging; the `LightweightMonitor` is for *production* training.

---

## Summary

| Concept | Key detail |
|---|---|
| `register_forward_hook` | Fires after `module.forward()`. Captures activations. Always `.detach()` the output. |
| `register_full_backward_hook` | Fires during backward. `grad_output[0]` = gradient flowing into this module. |
| `register_hook` on parameter | Fires after gradient accumulation, before optimizer step. |
| Always remove hooks | Store handles, call `.remove()`. Forgotten hooks are memory leaks and slow training. |
| Gradient-to-weight ratio | $\rho_l = \|\nabla W_l\| / \|W_l\|$. Healthy range: $[10^{-3}, 10^{-2}]$. |
| Lever 1 — Init | Too small: weights don't move. Too large: saturation, vanishing grads. |
| Lever 2 — Layer Norm | Without LN: activation scale compounds across depth. Ratio degrades with layer index. |
| Lever 3 — LR | Does not appear in $\rho$ directly — appears in effective update ratio $\text{lr} \cdot \rho$. |
| Lever 4 — Grad clipping | Pre-clip norm is the key signal. Consistently at clip threshold → reduce LR or raise clip. |
| Lever 5 — Residual scaling | $1/\sqrt{2L}$ on output projections. Without it, ratio degrades monotonically with depth. |
| Dead neurons | `act_frac_zero` > 0.5 for a layer → most neurons dead. Not recoverable via SGD. |
| GELU dead proxy | Use `abs(act) < 0.01` fraction — GELU never produces exactly zero. |
| Production monitoring | Read `.grad` directly after backward. No hooks needed. Cheap enough for every run. |

---

## Exercises

**1.** Modify `GradientMonitor` to also record the histogram of gradient values (as a `torch.Tensor` of bin counts) for each layer. Plot the histogram at steps 0, 50, 100, 150, and 200 for the `blocks.0.attn.W_o` layer. Observe how the distribution changes as training progresses.

**2.** Reproduce the dead neuron experiment deliberately: train with `lr=1e-1` (very high) for 50 steps on a 2-layer FFN with ReLU (not GELU). Plot `act_frac_zero` over time. Then reduce LR to `1e-3` and continue — confirm the dead neurons do not recover.

**3.** The `LightweightMonitor` reads `.grad` directly which is the post-accumulation, pre-optimizer gradient. Add a comparison: also compute the ratio using `param.grad` for the same layers as the full `GradientMonitor`. Verify they agree at the same step.

**4.** The five-lever combined chart uses mean ratio across all layers. Rebuild it using the ratio of the *earliest* layer only (e.g., `blocks.0.attn.W_qkv`). Observe which levers affect early layers most severely — this is the hardest layer to train in any deep network.

**5.** Add a `trigger_alert` argument to `GradientMonitor.record()` that calls a user-provided callback whenever any layer's ratio falls outside `[lo, hi]`. Use this to automatically checkpoint the model when a ratio spike is detected — the checkpoint captures the model state just before instability.

**6.** The gradient-to-weight ratio uses L2 norms. Implement an alternative using the L∞ norm (`abs().max()`) for both numerator and denominator. Compare the two ratios on a healthy training run. Which is more sensitive to outlier gradients? Which would you prefer for early warning of gradient spikes?